# Pairwise Verdict Browser

Interactive browser for the ties / wins / losses produced by the 100-sample dry-run of
`mistral_r16_a16_e1_b16_w30` vs `baseline`.

Uses ipywidgets dropdowns to filter by **judge** (j1/j2/agreement), **dimension**
(quality/instruction), and **verdict** (finetuned / tie / baseline). Each row shows
the prompt, both responses side-by-side, and both judges' justifications.

Source files (must exist in `outputs/eval_dry_run/`):
- `mistral_r16_a16_e1_b16_w30_pairwise.jsonl`
- `mistral_r16_a16_e1_b16_w30_row_scores.jsonl`
- `baseline_row_scores.jsonl` *(optional — used for absolute-score delta display)*

In [2]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from ipywidgets import Dropdown, IntSlider, Checkbox, HBox, VBox, Output, interactive_output

DRY_DIR = Path("outputs/eval_dry_run")
MODEL_LABEL = "mistral_r16_a16_e1_b16_w30"

pairs = pd.DataFrame([json.loads(l) for l in (DRY_DIR / f"{MODEL_LABEL}_pairwise.jsonl").open()])
ft_scores = pd.DataFrame([json.loads(l) for l in (DRY_DIR / f"{MODEL_LABEL}_row_scores.jsonl").open()])

bl_path = DRY_DIR / "baseline_row_scores.jsonl"
bl_scores = (
    pd.DataFrame([json.loads(l) for l in bl_path.open()]) if bl_path.exists() else None
)

print(f"pairs:        {len(pairs):4d} rows")
print(f"ft scores:    {len(ft_scores):4d} rows")
print(f"bl scores:    {len(bl_scores) if bl_scores is not None else 'n/a':>4} rows")
pairs.head(1).T

pairs:         100 rows
ft scores:     100 rows
bl scores:    1500 rows


,0
input,Schrijf een korte alinea over het gegeven onde...
baseline_output,Het gebruik van hernieuwbare energie is van gr...
finetuned_output,Hernieuwbare energie is essentieel voor een du...
j1_pairwise_quality_winner,tie
j1_pairwise_quality_winner_ab,tie
j1_pairwise_quality_winner_ba,tie
j1_pairwise_quality_justification_ab,Beide antwoorden zijn grammaticaal correct en ...
j1_pairwise_quality_justification_ba,Beide reacties zijn grammaticaal correct en na...
j1_pairwise_quality_flipped,False
j1_pairwise_instruction_winner,tie


## Quick verdict counts

In [3]:
summary_rows = []
for judge in ("j1", "j2"):
    for dim in ("quality", "instruction"):
        col = f"{judge}_pairwise_{dim}_winner"
        if col not in pairs.columns:
            continue
        counts = pairs[col].value_counts().to_dict()
        flip_col = f"{judge}_pairwise_{dim}_flipped"
        flip_rate = (
            pairs[flip_col].astype(bool).mean() if flip_col in pairs.columns else None
        )
        summary_rows.append({
            "judge": judge,
            "dim": dim,
            "finetuned": counts.get("finetuned", 0),
            "tie": counts.get("tie", 0),
            "baseline": counts.get("baseline", 0),
            "flip_rate": round(flip_rate, 3) if flip_rate is not None else None,
        })

pd.DataFrame(summary_rows)

,judge,dim,finetuned,tie,baseline,flip_rate
0,j1,quality,30,43,27,0.30
1,j1,instruction,7,51,42,0.10
2,j2,quality,39,36,25,0.07
3,j2,instruction,11,77,12,0.02


## Inter-judge agreement matrix

Where do judges agree, where do they disagree? Diagonal = agreement.

In [4]:
for dim in ("quality", "instruction"):
    j1c = f"j1_pairwise_{dim}_winner"
    j2c = f"j2_pairwise_{dim}_winner"
    if j1c not in pairs.columns or j2c not in pairs.columns:
        continue
    mat = pd.crosstab(pairs[j1c], pairs[j2c], rownames=[f"j1 {dim}"], colnames=[f"j2 {dim}"])
    display(mat)

j2 quality,baseline,finetuned,tie
j1 quality,,,
baseline,13,4,10
finetuned,4,19,7
tie,8,16,19


j2 instruction,baseline,finetuned,tie
j1 instruction,,,
baseline,12,1,29
finetuned,0,5,2
tie,0,5,46


## Interactive browser

Pick a judge, dimension and verdict. Step through matching rows with the slider.

- **judge = both_agree** → only show rows where j1 and j2 returned the same verdict
- **judge = disagree** → only show rows where j1 and j2 returned different verdicts
- **only_flipped** → restrict to rows where the judge's verdict changed on A/B swap

In [ ]:
def filter_rows(judge: str, dim: str, verdict: str, only_flipped: bool) -> pd.DataFrame:
    df = pairs.copy()
    j1c = f"j1_pairwise_{dim}_winner"
    j2c = f"j2_pairwise_{dim}_winner"

    if judge == "both_agree":
        df = df[df[j1c] == df[j2c]]
        if verdict != "any":
            df = df[df[j1c] == verdict]
    elif judge == "disagree":
        df = df[df[j1c] != df[j2c]]
    else:
        col = f"{judge}_pairwise_{dim}_winner"
        if verdict != "any":
            df = df[df[col] == verdict]
        if only_flipped:
            fcol = f"{judge}_pairwise_{dim}_flipped"
            if fcol in df.columns:
                df = df[df[fcol].astype(bool)]
    return df.reset_index(drop=True)


def _esc(s):
    return (s or "").replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def _abs_score(scores_df, input_text, judge, dim_field):
    if scores_df is None:
        return None
    col = f"{judge}_{dim_field}"
    if col not in scores_df.columns:
        return None
    hit = scores_df[scores_df["input"] == input_text]
    if hit.empty:
        return None
    val = hit.iloc[0][col]
    return None if pd.isna(val) else val


def render_row(row: pd.Series, dim: str) -> str:
    """Render one pair as side-by-side HTML."""
    j1v = row[f"j1_pairwise_{dim}_winner"]
    j2v = row[f"j2_pairwise_{dim}_winner"]

    # Pick the most informative absolute proxy for this dimension
    abs_fields = {
        "quality": [("grammar_score", "0-5"), ("fluency_score", "0-5"), ("vocabulary_score", "0-2")],
        "instruction": [("instruction_following_score", "FT:0-3 / BL:0-5"), ("correctness_score", "0-5")],
    }[dim]

    score_rows = []
    for field, scale in abs_fields:
        cells = [f"<td><b>{field}</b> <small>({scale})</small></td>"]
        for judge in ("j1", "j2"):
            ft = _abs_score(ft_scores, row["input"], judge, field)
            bl = _abs_score(bl_scores, row["input"], judge, field)
            cells.append(f"<td>{judge} → ft={ft} bl={bl}</td>")
        score_rows.append("<tr>" + "".join(cells) + "</tr>")

    flip_j1 = row.get(f"j1_pairwise_{dim}_flipped", False)
    flip_j2 = row.get(f"j2_pairwise_{dim}_flipped", False)

    color = {"finetuned": "#2ca02c", "baseline": "#d62728", "tie": "#777"}

    html = f"""
    <div style="border:1px solid #ccc;border-radius:6px;padding:14px;margin:10px 0;font-family:system-ui">
      <div style="display:flex;gap:18px;align-items:center;margin-bottom:8px;flex-wrap:wrap">
        <span style="font-size:13px;color:#666">dim=<b>{dim}</b></span>
        <span>j1: <b style="color:{color.get(j1v, '#000')}">{j1v}</b>{' ⟳' if flip_j1 else ''}</span>
        <span>j2: <b style="color:{color.get(j2v, '#000')}">{j2v}</b>{' ⟳' if flip_j2 else ''}</span>
        <span style="color:{'#2ca02c' if j1v == j2v else '#d62728'};font-weight:600">{'AGREE' if j1v == j2v else 'DISAGREE'}</span>
      </div>
      <div style="background:#f7f7f7;padding:8px 12px;border-radius:4px;margin-bottom:8px">
        <small style="color:#666">PROMPT</small><br>
        <pre style="white-space:pre-wrap;margin:4px 0;font-family:inherit">{_esc(row['input'])}</pre>
      </div>
      <table style="width:100%;table-layout:fixed;border-collapse:collapse">
        <tr>
          <th style="text-align:left;width:50%;padding:6px;background:#eef;border:1px solid #dde">BASELINE</th>
          <th style="text-align:left;width:50%;padding:6px;background:#efe;border:1px solid #ded">FINETUNED</th>
        </tr>
        <tr>
          <td style="padding:8px;vertical-align:top;border:1px solid #dde">
            <pre style="white-space:pre-wrap;margin:0;font-family:inherit">{_esc(row['baseline_output'])}</pre>
          </td>
          <td style="padding:8px;vertical-align:top;border:1px solid #ded">
            <pre style="white-space:pre-wrap;margin:0;font-family:inherit">{_esc(row['finetuned_output'])}</pre>
          </td>
        </tr>
      </table>
      <details style="margin-top:8px">
        <summary style="cursor:pointer;font-weight:600">Judge justifications</summary>
        <div style="padding:8px;background:#fafafa;border-radius:4px;margin-top:4px">
          <p><b>j1 (A/B):</b> {_esc(row.get(f'j1_pairwise_{dim}_justification_ab', ''))}</p>
          <p><b>j1 (B/A):</b> {_esc(row.get(f'j1_pairwise_{dim}_justification_ba', ''))}</p>
          <p><b>j2 (A/B):</b> {_esc(row.get(f'j2_pairwise_{dim}_justification_ab', ''))}</p>
          <p><b>j2 (B/A):</b> {_esc(row.get(f'j2_pairwise_{dim}_justification_ba', ''))}</p>
        </div>
      </details>
      <details style="margin-top:6px">
        <summary style="cursor:pointer;font-weight:600">Absolute scores</summary>
        <table style="margin-top:4px;font-size:13px">
          {''.join(score_rows)}
        </table>
      </details>
    </div>
    """
    return html


judge_w = Dropdown(options=["j1", "j2", "both_agree", "disagree"], value="j2", description="judge")
dim_w = Dropdown(options=["quality", "instruction"], value="quality", description="dim")
verdict_w = Dropdown(options=["any", "finetuned", "tie", "baseline"], value="tie", description="verdict")
flipped_w = Checkbox(value=False, description="only flipped")
idx_w = IntSlider(value=0, min=0, max=0, step=1, description="row", continuous_update=False)
out = Output()

_state = {"df": pd.DataFrame()}


def _refilter(*_):
    df = filter_rows(judge_w.value, dim_w.value, verdict_w.value, flipped_w.value)
    _state["df"] = df
    idx_w.max = max(len(df) - 1, 0)
    if idx_w.value > idx_w.max:
        idx_w.value = 0
    _render()


def _render(*_):
    df = _state["df"]
    out.clear_output(wait=True)
    with out:
        if df.empty:
            display(HTML("<i>No rows match the current filter.</i>"))
            return
        i = min(idx_w.value, len(df) - 1)
        display(HTML(f"<b>{i + 1} / {len(df)}</b> matching rows"))
        display(HTML(render_row(df.iloc[i], dim_w.value)))


for w in (judge_w, dim_w, verdict_w, flipped_w):
    w.observe(_refilter, names="value")
idx_w.observe(_render, names="value")

_refilter()
display(VBox([HBox([judge_w, dim_w, verdict_w, flipped_w]), idx_w, out]))

## Export the current filter

Useful when you want to share a subset (e.g. all rows where j1 picked baseline but j2 tied).

In [6]:
out_path = Path("outputs/eval_dry_run/_filtered_export.jsonl")
df = _state["df"]
if df.empty:
    print("Filter is empty — nothing to export.")
else:
    df.to_json(out_path, orient="records", lines=True, force_ascii=False)
    print(f"Wrote {len(df)} rows to {out_path}")

Wrote 36 rows to outputs/eval_dry_run/_filtered_export.jsonl
